# <font color = #699F04>**Pré-processamento dos Dados**</font>

Notebook de pré-processamento da base `pacigeral_2025.csv` para uso em modelos de machine learning e em análises de sobrevivência.

**Fluxo do notebook** 

Carregamento da base bruta ➤ Seleção dos dados ➤ Transformações e recálculo de variáveis ➤ Construção de `time` e `event` ➤ Verificações de consistência ➤ Divisão treino/teste ➤ Armazenamento



## <font color = #B88A00>**Configuração Inicial**</font>

### <font color = #007EF5>**Dependências**</font>

Dependências utilizadas neste notebook:
- `json`
- `numpy`
- `pandas`
- `IPython.display`
- `sklearn.model_selection.train_test_split`

### <font color = #007EF5>**Importações e Paths**</font>


In [1]:
import json

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split

RAW_DATA_PATH = '../complete/pacigeral_2025.csv'
TREATED_DATA_PATH = '../processed/Colorretal_tratado_2025.csv'
TRAIN_DATA_PATH = '../processed/train_data_2025.csv'
TEST_DATA_PATH = '../processed/test_data_2025.csv'
METADATA_PATH = '../processed/dataset_metadata_2025.json'


## <font color = #B88A00>**Tratamento dos Dados**</font>


### <font color = #007EF5>**Preparação dos Dados**</font>


In [2]:
def data_preparing(df):
    """
    Descrição:
      Prepara o conjunto de dados para análise, aplicando filtros e transformações.

    Parâmetros:
      df: pandas.DataFrame
        O conjunto de dados que será preparado.

    Retorno:
      pandas.DataFrame
        O conjunto de dados após as transformações.
    """

    # Cópia do DataFrame original
    df_aux = df.copy()

    # Redefinição dos índices
    df_aux = df_aux.reset_index(drop=True)

    # ============================ #
    # === SELEÇÕES NAS COLUNAS === #
    # ============================ #

    # Seleção 1 - Topografia Colorretal (C18, C19, C20)
    df_aux = df_aux[df_aux.TOPOGRUP.isin(['C18', 'C19', 'C20'])]

    # Seleção 2 - Residentes de SP
    df_aux = df_aux[df_aux.UFRESID == 'SP']

    # Seleção 3 - Casos com confirmação microscópica
    df_aux = df_aux[df_aux.BASEDIAG == 3]

    # Seleção 4 - ECGRUP em I, II, III e IV
    df_aux = df_aux[df_aux.ECGRUP.isin(['I', 'II', 'III', 'IV'])]

    # Seleção 5 - ANODIAG até 2019
    df_aux = df_aux[df_aux.ANODIAG <= 2019]

    # Seleção 6 - IDADE maior que 19
    df_aux = df_aux[df_aux.IDADE > 19]

    # Seleção 7 - CATEATEND com grupos SUS e sem informação
    df_aux = df_aux[df_aux.CATEATEND.isin([2, 9])]

    # ========================== #
    # === AJUSTE DAS COLUNAS === #
    # ========================== #

    # Ajuste 1 - Número da DRS
    drs_num = df_aux["DRS"].astype(str).str.extract(r"(\d+)")[0]
    df_aux["nDRS"] = pd.to_numeric(drs_num, errors="coerce")
    df_aux = df_aux[df_aux["nDRS"].notna()]
    df_aux["nDRS"] = df_aux["nDRS"].astype(int)

    # Ajuste 2 - Recalcular variáveis
    # Ajuste 2.1 - Colunas de data para o formato datetime
    list_datas = ['DTCONSULT', 'DTDIAG', 'DTTRAT', 'DTULTINFO']
    for col_data in list_datas:
        df_aux[col_data] = pd.to_datetime(df_aux[col_data], errors="coerce")

    # Ajuste 2.2 - Calcular a diferença entre as datas para criar novas variáveis
    df_aux['CONSDIAG'] = (df_aux.DTDIAG - df_aux.DTCONSULT).dt.days
    df_aux['TRATCONS'] = (df_aux.DTTRAT - df_aux.DTCONSULT).dt.days
    df_aux['DIAGTRAT'] = (df_aux.DTTRAT - df_aux.DTDIAG).dt.days
    df_aux['ULTIDIAG'] = (df_aux.DTULTINFO - df_aux.DTDIAG).dt.days

    # Ajuste 2.3 - Preencher os valores NaN com -1
    df_aux.fillna({'CONSDIAG': -1, 'TRATCONS': -1, 'DIAGTRAT': -1}, inplace=True)

    # Ajuste 2.4 - Implementar a categorização das variáveis calculadas
    df_aux['CONSDIAG_CAT'] = [3 if consdiag < 0 else 0 if consdiag <= 30 else 1 if consdiag <= 60 else 2 for consdiag in df_aux.CONSDIAG]
    df_aux['TRATCONS_CAT'] = [3 if tratcons < 0 else 0 if tratcons <= 60 else 1 if tratcons <= 90 else 2 for tratcons in df_aux.TRATCONS]
    df_aux['DIAGTRAT_CAT'] = [3 if diagtrat < 0 else 0 if diagtrat <= 60 else 1 if diagtrat <= 90 else 2 for diagtrat in df_aux.DIAGTRAT]

    # Ajuste 3 - Assistência de Alta Complexidade em Oncologia
    # Ajuste 3.1 - Agrupamento das colunas de HABILIT
    condlist = [
        df_aux['HABILIT'].isin([1, 3, 5, 12]),
        df_aux['HABILIT'].isin([6, 7]),
        df_aux['HABILIT'].isin([2, 9, 10, 15])
    ]

    # Ajuste 3.2 - Nome dos grupos
    choicelist = [
        0,  # UNACON sem Radioterapia
        1,  # CACON
        2   # UNACON com Radioterapia
    ]

    # Ajuste 3.3 - Tudo aquilo que não está na seleção
    default = -1

    # Ajuste 3.4 - Seleção das colunas de HABILIT e ajuste do nome das colunas
    df_aux['HABILIT'] = np.select(condlist, choicelist, default).astype(int)

    # Ajuste 3.5 - Excluir tudo aquilo que não é das seleções do Ajuste 3
    df_aux = df_aux[df_aux['HABILIT'] != -1]

    # ========================== #
    # === CRIAÇÃO DE COLUNAS === #
    # ========================== #

    # Criação 1 - Coluna para morfologia do câncer colorretal
    # Criação 1.1 - Atribuição dos valores aos códigos
    COD_ADENO, COD_SINETE, COD_MUCINOSO, COD_INDIFEREN = 81403, 84903, 84803, 80203

    # Criação 1.2 - Seleção dos códigos
    codigos = [COD_ADENO, COD_SINETE, COD_MUCINOSO, COD_INDIFEREN]
    df_aux = df_aux[df_aux['MORFO'].isin(codigos)]

    # Criação 1.3 - Criação dos grupos da coluna de morfologia
    morfo_map = {
        COD_ADENO: 'Adenocarcinoma',
        COD_SINETE: 'Anel de sinete',
        COD_MUCINOSO: 'Mucinoso',
        COD_INDIFEREN: 'Indiferenciado'
    }

    df_aux['MORFO_CAT'] = df_aux['MORFO'].map(morfo_map)

    # ================================= #
    # === COLUNAS A SEREM REMOVIDAS === #
    # ================================= #

    drop_cols = [
        'UFNASC', 'UFRESID', 'CIDADE', 'DTCONSULT', 'CLINICA', 'DTDIAG', 'BASEDIAG', 'DESCTOPO', 'MORFO', 'DESCMORFO',
        'T', 'N', 'M', 'PT', 'PN', 'PM', 'S', 'G', 'LOCALTNM', 'IDMITOTIC', 'PSA', 'GLEASON', 'OUTRACLA',
        'META01', 'META02', 'META03', 'META04', 'DTTRAT', 'NAOTRAT', 'TRATAMENTO', 'TRATHOSP', 'TRATFANTES', 'TRATFAPOS',
        'NENHUMANT', 'CIRURANT', 'RADIOANT', 'QUIMIOANT', 'HORMOANT', 'TMOANT', 'IMUNOANT', 'OUTROANT',
        'NENHUMAPOS', 'CIRURAPOS', 'RADIOAPOS', 'QUIMIOAPOS', 'HORMOAPOS', 'TMOAPOS', 'IMUNOAPOS', 'OUTROAPOS',
        'DTULTINFO', 'CICI', 'CICIGRUP', 'CICISUBGRU', 'LATERALI', 'INSTORIG', 'PERDASEG', 'ERRO', 'DTRECIDIVA',
        'RECNENHUM', 'RECLOCAL', 'RECREGIO', 'RECDIST', 'REC01', 'REC02', 'REC03', 'REC04', 'DSCINST', 'CIDO', 'DESCIDO',
        'HABILIT2', 'HABIT11', 'HABILIT1', 'CIDADEH', 'DRS', 'DRS_INST', 'CIDADE_INS',
        'NENHUM', 'CIRURGIA', 'RADIO', 'QUIMIO', 'HORMONIO', 'IMUNO', 'OUTROS', 'RRAS',
        'TMO', 'TOPOGRUP', 'EC', 'IBGE', 'IBGEATEN', 'RRAS_INST', 'FAIXAETAR', 'CONSDIAG', 'DIAGTRAT', 'TRATCONS'
    ]

    # Seleção final das colunas do novo DataFrame
    col = df_aux.columns.drop(drop_cols)

    return df_aux[col]


In [3]:
# Banco de Dados Original
df = pd.read_csv(RAW_DATA_PATH, low_memory=False)

# Exibição
display(df.head(3))
print(df.shape)

,INSTITU,ESCOLARI,IDADE,SEXO,UFNASC,UFRESID,IBGE,CIDADE,CATEATEND,DTCONSULT,...,DRS_INST,CIDADE_INS,CIDO,DESCIDO,RRAS_INST,HABILIT,HABIT11,HABILIT1,HABILIT2,CIDADEH
0,48593,1,0,1,SP,GO,5200050,ABADIA DE GOIAS,9,2008-11-17,...,DRS 07 Campinas,CAMPINAS,89003.0,"RABDOMIOSSARCOMA, SOE",15,15,UNACON exclusiva de Oncologia Pediátrica com S...,2,1,Campinas
1,9,1,0,1,MT,MT,5100102,ACORIZAL,9,2003-03-07,...,DRS 01 Grande Sao Paulo,SAO PAULO,95103.0,"RETINOBLASTOMA, SOE",6,15,UNACON exclusiva de Oncologia Pediátrica com S...,2,1,São Paulo
2,8672,1,0,1,AC,AC,1200013,ACRELANDIA,9,2006-06-15,...,DRS 01 Grande Sao Paulo,SAO PAULO,98613.0,"LEUCEMIA MIELOIDE AGUDA, SOE",6,7,CACON com Serviço de Oncologia Pediátrica,3,2,São Paulo


(1317509, 105)


In [4]:
# Banco de Dados Preparado
df_colo = data_preparing(df)

# Exibição
display(df_colo.head(3))
print(df_colo.shape)

,INSTITU,ESCOLARI,IDADE,SEXO,CATEATEND,DIAGPREV,TOPO,ECGRUP,ULTINFO,ANODIAG,HABILIT,nDRS,ULTIDIAG,CONSDIAG_CAT,TRATCONS_CAT,DIAGTRAT_CAT,MORFO_CAT
33774,9326,2,20,1,2,1,C182,IV,3,2019,1,9,966,2,2,0,Adenocarcinoma
33956,12,4,20,2,9,2,C209,IV,3,2008,2,7,95,3,0,0,Adenocarcinoma
34103,16624,5,20,2,2,2,C189,IV,3,2014,1,1,759,3,2,2,Adenocarcinoma


(32664, 17)


In [5]:
# Lista de colunas do DataFrame preparado
print(f"Colunas Preditoras: {len(df_colo.columns)}")
df_colo.columns.tolist()

Colunas Preditoras: 17


['INSTITU',
 'ESCOLARI',
 'IDADE',
 'SEXO',
 'CATEATEND',
 'DIAGPREV',
 'TOPO',
 'ECGRUP',
 'ULTINFO',
 'ANODIAG',
 'HABILIT',
 'nDRS',
 'ULTIDIAG',
 'CONSDIAG_CAT',
 'TRATCONS_CAT',
 'DIAGTRAT_CAT',
 'MORFO_CAT']

### <font color = #007EF5>**Variáveis de Sobrevivência**</font>


In [6]:
def pred_cols(df_colo):
    """
    Descrição:
      Formatação das colunas que são usadas para a predição.

    Parâmetros:
      df_colo: pandas.DataFrame
        O conjunto de dados já preparado.

    Retorno:
      pandas.DataFrame
        O conjunto de dados com as colunas de predição formatadas.
    """

    # Cópia do DataFrame
    df_aux = df_colo.copy()

    # Redefinição dos índices
    df_aux = df_aux.reset_index(drop=True)

    # ===================== #
    # === ULTIDIAG/TIME === #
    # ===================== #

    # Eliminação das linhas onde 'ULTIDIAG' é menor que zero
    df_aux = df_aux[df_aux.ULTIDIAG >= 0]

    # Dividir a coluna de tempo ('ULTIDIAG') em meses
    df_aux['time'] = np.ceil(df_aux['ULTIDIAG'] / 30).astype(int)

    # Ajustar valores menores ou iguais a zero para 1
    df_aux.loc[df_aux['time'] == 0, 'time'] = 1

    # Eliminação da coluna 'ULTIDIAG' (já transformada em 'time')
    df_aux = df_aux.drop('ULTIDIAG', axis=1)

    # ===================== #
    # === ULTINFO/EVENT === #
    # ===================== #

    # Ajuste da coluna 'ULTINFO' para ser binária (1 = morte / 0 = vivo)
    df_aux['event'] = df_aux['ULTINFO'].apply(lambda x: 1 if x in [3, 4] else 0)

    # Ajustar valores maiores que 60 meses para 61 e definir o evento como 0
    df_aux.loc[df_aux['time'] > 60, ['time', 'event']] = [61, 0]

    # Eliminar a coluna 'ULTINFO' (agora transformada para 'event')
    df_aux = df_aux.drop('ULTINFO', axis=1)

    return df_aux

In [7]:
# Aplicação da preparação das colunas preditivas
df_colo = pred_cols(df_colo)

#### <font color = #FF120A>**Evento**</font>

In [8]:
# Visualização das colunas com o evento
display(df_colo['event'].value_counts().sort_index())

event
0    14524
1    18140
Name: count, dtype: int64

In [9]:
# Análise exploratória da distribuição do evento
print(f"\nDistribuição do Evento:")
print(f"  Eventos (morte): {df_colo['event'].sum():,} ({df_colo['event'].mean()*100:.1f}%)")
print(f"  Censurados: {(df_colo['event']==0).sum():,} ({(1-df_colo['event'].mean())*100:.1f}%)")
print(f"  Mínimo: {df_colo['time'].min()}")
print(f"  Máximo: {df_colo['time'].max()}")
print(f"  Média: {df_colo['time'].mean():.2f}")
print(f"  Mediana: {df_colo['time'].median():.0f}")


Distribuição do Evento:
  Eventos (morte): 18,140 (55.5%)
  Censurados: 14,524 (44.5%)
  Mínimo: 1
  Máximo: 61
  Média: 35.83
  Mediana: 36


#### <font color = #FF120A>**Tempo**</font>

In [10]:
# Visualização da coluna de tempo
display(df_colo['time'].value_counts().sort_index())

time
1      1188
2      1007
3       884
4       812
5       714
      ...  
57      141
58      141
59      140
60      158
61    12412
Name: count, Length: 61, dtype: int64

In [11]:
print(f"\nDistribuição do Tempo (meses):")
print(f"  Mínimo: {df_colo['time'].min()}")
print(f"  Máximo: {df_colo['time'].max()}")


Distribuição do Tempo (meses):
  Mínimo: 1
  Máximo: 61


## <font color = #B88A00>**Salvamento da Base Tratada**</font>


In [12]:
df_colo.to_csv(TREATED_DATA_PATH, index=False)

## <font color = #B88A00>**Preparação Final para Modelagem**</font>


### <font color = #007EF5>**Definição de Variáveis Alvo e Preditoras**</font>


In [13]:
# Lista de colunas do DataFrame preparado
survival_cols = ['time', 'event']
# Remover as colunas de tempo e evento para obter apenas as colunas preditoras
predictor_cols = [col for col in df_colo.columns if col not in survival_cols]

print(f"Colunas Alvo: {len(survival_cols)}")
print(f"Alvo: {survival_cols}\n")
print(f"Colunas Preditoras: {len(predictor_cols)}")
print(f"Preditoras: {predictor_cols}")

Colunas Alvo: 2
Alvo: ['time', 'event']

Colunas Preditoras: 15
Preditoras: ['INSTITU', 'ESCOLARI', 'IDADE', 'SEXO', 'CATEATEND', 'DIAGPREV', 'TOPO', 'ECGRUP', 'ANODIAG', 'HABILIT', 'nDRS', 'CONSDIAG_CAT', 'TRATCONS_CAT', 'DIAGTRAT_CAT', 'MORFO_CAT']


### <font color = #007EF5>**Verificações de Consistência**</font>


In [14]:
# Análise dos valores nulos
missing = df_colo[predictor_cols + survival_cols].isnull().sum().sum()
print(f"Valores Ausentes: {missing}")

print(f"\nTipos de Dados:")
print(df_colo[predictor_cols].dtypes.value_counts())

print(f"\nColunas Categóricas:")
print(df_colo[predictor_cols].select_dtypes(include='object').columns.tolist())

Valores Ausentes: 0

Tipos de Dados:
int64     12
object     3
Name: count, dtype: int64

Colunas Categóricas:
['TOPO', 'ECGRUP', 'MORFO_CAT']


### <font color = #007EF5>**Divisão Train/Test (80/20)**</font>


In [15]:
# Divisão do dataset em treino e teste (80% treino, 20% teste) com estratificação pelo evento
df_train, df_test = train_test_split(df_colo, test_size=0.20, random_state=19, stratify=df_colo['event'])

# Exibição das formas dos datasets e da taxa de eventos
print(f"Dataset de Treinamento: {df_train.shape}")
print(f"Dataset de Teste: {df_test.shape}")
print(f"\nTaxa de eventos no Treino: {df_train['event'].mean():.1%}")
print(f"Taxa de eventos no Teste: {df_test['event'].mean():.1%}")

Dataset de Treinamento: (26131, 17)
Dataset de Teste: (6533, 17)

Taxa de eventos no Treino: 55.5%
Taxa de eventos no Teste: 55.5%


### <font color = #007EF5>**Salvamento de Datasets e Metadados**</font>


#### <font color = #FF120A>**Treino e Teste**</font>

In [16]:
# Salvar os datasets de treino e teste em arquivos CSV
df_train.to_csv(TRAIN_DATA_PATH, index=False)
df_test.to_csv(TEST_DATA_PATH, index=False)

#### <font color = #FF120A>**Metadados**</font>

In [17]:
# Gerar e salvar o arquivo de metadados do dataset
metadata = {
    'total_samples': len(df_colo),
    'train_samples': len(df_train),
    'test_samples': len(df_test),
    'n_predictors': len(predictor_cols),
    'predictor_columns': predictor_cols,
    'target_columns': survival_cols,
    'total_events': int(df_colo['event'].sum()),
    'event_rate': float(df_colo['event'].mean()),
    'time_min_months': int(df_colo['time'].min()),
    'time_max_months': int(df_colo['time'].max()),
    'train_event_rate': float(df_train['event'].mean()),
    'test_event_rate': float(df_test['event'].mean())
}
# Salvar os metadados em um arquivo JSON
with open(METADATA_PATH, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)